In [1]:
# ============================================================
# FUNCTION 7 - WEEK 13 FINAL-WEEK BAYESIAN OPTIMISATION
# Run from inside week13/
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)


# ------------------------------------------------------------
# 1. Load Week 13 cumulative data
# ------------------------------------------------------------

X = np.load(
    "function7/initial_inputs.npy"
)

Y = np.load(
    "function7/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

tree = cKDTree(X)

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 12 calibration check
# ------------------------------------------------------------
#
# Approximate Week 12 GP prediction:
#
# mean ≈ 3.074907
# std  ≈ 0.009318
#
# Actual Week 12 result:
#
# 3.07705726187221
#
# The realised result was only about +0.23 sigma from the
# prediction, giving strong local calibration evidence.
# ------------------------------------------------------------

week12_actual = (
    3.07705726187221
)

week12_pred_mean = (
    3.074907
)

week12_pred_std = (
    0.009318
)

week12_error = (
    week12_actual
    - week12_pred_mean
)

week12_z_error = (
    week12_error
    / week12_pred_std
)

print("\n================================")
print("WEEK 12 CALIBRATION CHECK")
print("================================")

print(
    "Approx predicted mean:",
    week12_pred_mean
)

print(
    "Approx predicted std :",
    week12_pred_std
)

print(
    "Actual              :",
    week12_actual
)

print(
    "\nPrediction error:",
    week12_error
)

print(
    "\nError / predicted std:",
    week12_z_error
)


# ------------------------------------------------------------
# 3. Recent local evidence
# ------------------------------------------------------------

week11_x = np.array([
    0.174931,
    0.257671,
    0.577020,
    0.262318,
    0.287196,
    0.669103
])

week11_y = (
    3.067532247418812
)

week12_x = np.array([
    0.165836,
    0.256907,
    0.556250,
    0.254875,
    0.282301,
    0.673252
])

week12_y = (
    3.07705726187221
)

print("\n================================")
print("RECENT LOCAL EVIDENCE")
print("================================")

print("\nWeek 11:")
print(
    week11_x,
    "->",
    week11_y
)

print("\nWeek 12 / incumbent:")
print(
    week12_x,
    "->",
    week12_y
)

print(
    "\nWeek 11 -> Week 12 displacement:"
)

print(
    week12_x
    - week11_x
)

print(
    "\nDistance Week 11 -> Week 12:"
)

print(
    np.linalg.norm(
        week12_x
        - week11_x
    )
)

print(
    "\nObjective improvement:"
)

print(
    week12_y
    - week11_y
)


# ------------------------------------------------------------
# 4. Fit updated ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(6) * 0.3,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=15,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)

print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)

print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Final week:
# tighter exploitation around proven Week 12 basin
empirical_cap = min(
    1.0 * nearest_distance,
    0.025
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.08 * lengthscales,
    0.010,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 13 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)

print(
    trust_half_width
)

print(
    "\nLower:"
)

print(
    lower
)

print(
    "\nUpper:"
)

print(
    upper
)


# ------------------------------------------------------------
# 8. Dense local candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

candidates = rng.uniform(
    lower,
    upper,
    size=(
        300000,
        6
    )
)

# Portal precision before duplicate filtering
candidates = np.round(
    candidates,
    6
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after portal rounding + duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 10. EI - diagnostic only
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI - DIAGNOSTIC ONLY")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 11. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"\nbeta={beta}"
    )

    print(
        "candidate =",
        candidates[idx]
    )

    print(
        "mean =",
        mu[idx]
    )

    print(
        "std =",
        sigma[idx]
    )

    print(
        "distance =",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(
        len(x)
    ):

        if abs(
            x[j]
            - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j]
            - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )


# ------------------------------------------------------------
# 14. Axis profile diagnostics
# ------------------------------------------------------------

print("\n================================")
print("AXIS PROFILE DIAGNOSTICS")
print("================================")

for j in range(6):

    values = np.linspace(
        lower[j],
        upper[j],
        40000
    )

    profile = np.tile(
        best_x,
        (
            len(values),
            1
        )
    )

    profile[:, j] = values

    profile = np.round(
        profile,
        6
    )

    profile_distance, _ = tree.query(
        profile,
        k=1
    )

    profile_valid = profile[
        profile_distance > 0.01
    ]

    if len(profile_valid) == 0:

        print(
            f"\nx{j+1}: no admissible points"
        )

        continue

    profile_mu, profile_sigma = gp.predict(
        profile_valid,
        return_std=True
    )

    idx = np.argmax(
        profile_mu
    )

    print(
        f"\nx{j+1} profile best:"
    )

    print(
        "candidate =",
        profile_valid[idx]
    )

    print(
        "mean =",
        profile_mu[idx]
    )

    print(
        "std =",
        profile_sigma[idx]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            profile_valid[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 15. Week 11 -> Week 12 continuation direction
# ------------------------------------------------------------
#
# Week 12 improved on Week 11, so unlike F6 we explicitly test
# continuation in the SAME empirically successful direction.
# ------------------------------------------------------------

direction = (
    best_x
    - week11_x
)

direction_norm = np.linalg.norm(
    direction
)

direction = (
    direction
    / direction_norm
)

positive_limits = []

for j in range(6):

    if direction[j] > 0:

        positive_limits.append(
            (
                upper[j]
                - best_x[j]
            )
            / direction[j]
        )

    elif direction[j] < 0:

        positive_limits.append(
            (
                lower[j]
                - best_x[j]
            )
            / direction[j]
        )

t_max = min(
    positive_limits
)

t_values = np.linspace(
    0.0,
    t_max,
    60000
)

continuation_candidates = (
    best_x
    + t_values[:, None]
    * direction
)

continuation_candidates = np.round(
    continuation_candidates,
    6
)

continuation_distance, _ = tree.query(
    continuation_candidates,
    k=1
)

continuation_valid = continuation_candidates[
    continuation_distance > 0.01
]

print("\n================================")
print("WEEK 11 -> WEEK 12 CONTINUATION")
print("================================")

if len(continuation_valid) > 0:

    continuation_mu, continuation_sigma = gp.predict(
        continuation_valid,
        return_std=True
    )

    continuation_idx = np.argmax(
        continuation_mu
    )

    print(
        "Best continuation candidate =",
        continuation_valid[
            continuation_idx
        ]
    )

    print(
        "mean =",
        continuation_mu[
            continuation_idx
        ]
    )

    print(
        "std =",
        continuation_sigma[
            continuation_idx
        ]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            continuation_valid[
                continuation_idx
            ]
            - best_x
        )
    )

else:

    print(
        "No admissible continuation candidates."
    )


# ------------------------------------------------------------
# 16. Exact one-step continuation diagnostic
# ------------------------------------------------------------
#
# Continue the Week11 -> Week12 displacement once more.
# This is diagnostic only; it will not automatically be chosen.
# ------------------------------------------------------------

one_step_candidate = (
    best_x
    + (
        best_x
        - week11_x
    )
)

one_step_candidate = np.clip(
    one_step_candidate,
    0.0,
    1.0
)

one_step_candidate = np.round(
    one_step_candidate,
    6
)

one_step_dist, one_step_idx = tree.query(
    one_step_candidate.reshape(1, -1),
    k=1
)

print("\n================================")
print("ONE-STEP CONTINUATION")
print("================================")

print(
    "candidate =",
    one_step_candidate
)

print(
    "nearest observed distance =",
    one_step_dist[0]
)

print(
    "nearest observed point =",
    X[
        one_step_idx[0]
    ]
)

if one_step_dist[0] > 0.01:

    one_mu, one_sigma = gp.predict(
        one_step_candidate.reshape(1, -1),
        return_std=True
    )

    print(
        "mean =",
        one_mu[0]
    )

    print(
        "std =",
        one_sigma[0]
    )

    print(
        "distance from incumbent =",
        np.linalg.norm(
            one_step_candidate
            - best_x
        )
    )

else:

    print(
        "One-step continuation is too close to an existing observation."
    )


# ------------------------------------------------------------
# 17. Closest admissible continuation point
# ------------------------------------------------------------

if len(continuation_valid) > 0:

    continuation_incumbent_dist = np.linalg.norm(
        continuation_valid
        - best_x,
        axis=1
    )

    closest_idx = np.argmin(
        continuation_incumbent_dist
    )

    closest_candidate = (
        continuation_valid[
            closest_idx
        ]
    )

    closest_mu, closest_sigma = gp.predict(
        closest_candidate.reshape(1, -1),
        return_std=True
    )

    print("\n================================")
    print("CLOSEST ADMISSIBLE CONTINUATION")
    print("================================")

    print(
        "candidate =",
        closest_candidate
    )

    print(
        "mean =",
        closest_mu[0]
    )

    print(
        "std =",
        closest_sigma[0]
    )

    print(
        "distance from incumbent =",
        continuation_incumbent_dist[
            closest_idx
        ]
    )

DATA
X shape: (42, 6)
Y shape: (42,)

Current best:
[0.165836 0.256907 0.55625  0.254875 0.282301 0.673252] -> 3.07705726187221

Y range:
min = 0.0027014650245082332
max = 3.07705726187221
std = 1.0936302077919275

WEEK 12 CALIBRATION CHECK
Approx predicted mean: 3.074907
Approx predicted std : 0.009318
Actual              : 3.07705726187221

Prediction error: 0.002150261872210013

Error / predicted std: 0.23076431339450668

RECENT LOCAL EVIDENCE

Week 11:
[0.174931 0.257671 0.57702  0.262318 0.287196 0.669103] -> 3.067532247418812

Week 12 / incumbent:
[0.165836 0.256907 0.55625  0.254875 0.282301 0.673252] -> 3.07705726187221

Week 11 -> Week 12 displacement:
[-0.009095 -0.000764 -0.02077  -0.007443 -0.004895  0.004149]

Distance Week 11 -> Week 12:
0.024723856818870252

Objective improvement:
0.00952501445339804


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.626**2 * Matern(length_scale=[0.523, 0.441, 0.826, 0.434, 0.281, 0.41], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[0.52326428 0.44124106 0.82594798 0.43426801 0.28078115 0.41006921]

Normalised inverse-lengthscale sensitivity:
[0.13958685 0.16553494 0.0884327  0.16819294 0.26013432 0.17811825]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.024723856818870252

Empirical cap: 0.024723856818870252

WEEK 13 TRUST REGION
Centre:
[0.165836 0.256907 0.55625  0.254875 0.282301 0.673252]

Half-widths:
[0.02472386 0.02472386 0.02472386 0.02472386 0.02246249 0.02472386]

Lower:
[0.14111214 0.23218314 0.53152614 0.23015114 0.25983851 0.64852814]

Upper:
[0.19055986 0.28163086 0.58097386 0.27959886 0.30476349 0.69797586]

Candidates after portal rounding + duplicate filtering:
299786

PRIMARY EI - DIAGNOSTIC ONLY
candidate = [0.185148 0.23353  0.533834 0.247033 0.297018 0.679719]
mean = 3.063808408091786
std = 0.05609057866418857
EI = 0.0163738233410177

In [2]:
# ============================================================
# FINAL FUNCTION 7 - WEEK 13 SELECTION
# ============================================================
#
# FINAL CAPSTONE QUERY
#
# Evidence:
# - Week 12 produced a new observed best:
#
#       3.07705726187221
#
# - Week 12 was also very well calibrated:
#
#       prediction error ≈ +0.23 sigma
#
#   giving strong evidence that the local GP is reliable in
#   this basin.
#
# - The updated highest-posterior-mean candidate predicts:
#
#       mean = 3.079561
#       std  = 0.011516
#
#   which is above the current incumbent.
#
# - It lies only ~0.01029 from the incumbent, making it a
#   highly local exploitation move while safely exceeding
#   the 0.01 duplicate threshold.
#
# - Low-beta UCB moves about twice as far from the incumbent,
#   while EI / higher-beta UCB increasingly favour uncertainty.
#
# - Simply continuing the Week11 -> Week12 direction is
#   predicted below the incumbent, so we use the updated local
#   posterior optimum rather than blindly extrapolating.
#
# Final-week strategy:
# exploit the exceptionally well-calibrated local posterior
# with the highest predicted mean.
# ============================================================

week13_candidate = np.array([
    0.165154,
    0.250080,
    0.549909,
    0.253314,
    0.284687,
    0.676488
])

print("================================")
print("FUNCTION 7 - WEEK 13 FINAL")
print("================================")

print("\nCandidate:")
print(week13_candidate)


# ------------------------------------------------------------
# GP prediction
# ------------------------------------------------------------

final_mu, final_sigma = gp.predict(
    week13_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])


# ------------------------------------------------------------
# Distance from incumbent
# ------------------------------------------------------------

distance_from_best = np.linalg.norm(
    week13_candidate - best_x
)

print("\nDistance from current best:")
print(distance_from_best)


# ------------------------------------------------------------
# Duplicate-rule verification
# ------------------------------------------------------------

nearest_dist, nearest_idx = tree.query(
    week13_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

print("\nNearest observed point:")
print(X[nearest_idx[0]])

print("\nDuplicate rule satisfied:")
print(
    nearest_dist[0] > 0.01
)


# ------------------------------------------------------------
# Expected improvement over incumbent prediction
# ------------------------------------------------------------

print("\nPredicted improvement over current best:")
print(
    final_mu[0] - best_y
)


# ------------------------------------------------------------
# Portal format
# ------------------------------------------------------------

portal = "-".join(
    f"{x:.6f}"
    for x in week13_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 7 - WEEK 13 FINAL

Candidate:
[0.165154 0.25008  0.549909 0.253314 0.284687 0.676488]

Predicted mean:
3.079561418207245

Predicted std:
0.011515630553908893

Distance from current best:
0.010289934256349765

Nearest observed-point distance:
0.010289934256349763

Nearest observed point:
[0.165836 0.256907 0.55625  0.254875 0.282301 0.673252]

Duplicate rule satisfied:
True

Predicted improvement over current best:
0.0025041563350347573

Portal format:
0.165154-0.250080-0.549909-0.253314-0.284687-0.676488
